# Объяснятель: выравнивание Qwen3-4B-Instruct-2507

Полностью воспроизводимый Colab-пайплайн: SFT → style-DPO, отдельная Reward Model, затем независимые quality-DPO и SimPO.

## 0. Цель и инварианты

- Все random seeds равны `42`.
- Обучение выполняется через QLoRA 4-bit на T4.
- `public_test_*` не участвуют в обучении.
- Style-генерация выполняется с `do_sample=False`.
- Все пять метрик вычисляются внутри этого ноутбука.
- DPO и SimPO по качеству ветвятся от одного style-DPO чекпоинта.

## 1. Настройка окружения

In [ ]:
%pip install -q \
  transformers==4.57.3 trl==0.24.0 peft==0.17.1 \
  accelerate==1.11.0 bitsandbytes==0.48.2 datasets==4.3.0 \
  scikit-learn==1.7.2 safetensors==0.6.2

In [ ]:
import gc
import hashlib
import json
import os
import platform
import random
import shutil
import subprocess
import urllib.request
import zipfile
from collections import Counter
from pathlib import Path

import bitsandbytes as bnb
import datasets
import numpy as np
import peft
import sklearn
import torch
import transformers
import trl
from transformers import set_seed

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

assert torch.cuda.is_available(), "Для полного прогона требуется GPU T4"
print("Python:", platform.python_version())
print("GPU:", torch.cuda.get_device_name(0))
print("transformers:", transformers.__version__)
print("trl:", trl.__version__)
print("peft:", peft.__version__)
print("bitsandbytes:", bnb.__version__)
print("datasets:", datasets.__version__)
print("scikit-learn:", sklearn.__version__)
subprocess.run(["nvidia-smi"], check=True)

## 2. Данные и проверки

Загрузка официального архива, проверка SHA-256 и схем JSONL.

In [ ]:
ASSET_URL = "https://edu.tbank.ru/files/c1005bf0-8695-451a-9616-87c8687dce27"
RUNTIME_ROOT = Path("/content/alignment_explainer")
ARCHIVE_PATH = RUNTIME_ROOT / "ml-olympiad-red-task.zip"
EXTRACT_DIR = RUNTIME_ROOT / "source"
CHECKPOINT_DIR = RUNTIME_ROOT / "checkpoints"

RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

if not ARCHIVE_PATH.exists():
    print("Downloading organizer assets...")
    urllib.request.urlretrieve(ASSET_URL, ARCHIVE_PATH)

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True)
with zipfile.ZipFile(ARCHIVE_PATH) as archive:
    archive.extractall(EXTRACT_DIR)

kid_matches = list(EXTRACT_DIR.rglob("kid_adult.jsonl"))
assert len(kid_matches) == 1, f"Expected one kid_adult.jsonl, found {kid_matches}"
PROJECT_ROOT = kid_matches[0].parent.parent
DATA_DIR = PROJECT_ROOT / "data"
METRICS_DIR = PROJECT_ROOT / "metrics"
print("Assets root:", PROJECT_ROOT)

In [ ]:
EXPECTED_SHA256 = {
    "data/good_bad.jsonl": "bf50f3af0127df71d891c5a65eb75220104157f3e27b613aacbae1761c08998b",
    "data/kid_adult.jsonl": "52bacff1c6d5d50ca3dd473f8d754cf1dfcce7e02ecf162cda2c18719a138748",
    "data/public_test_quality.jsonl": "bc8b21bf04c88e99d420569c61f46309f71d04601159b80ea258760e8d871780",
    "data/public_test_style.jsonl": "d0f5fb848245b18e97b97fe5158c602f3f2b49b8ec6588f93a0f0e9f10c58efe",
    "metrics/style_clf.pkl": "b5cf7b53417033de19b9c44a43402bce0e6eeece44b1abac2cf596785b60888d",
    "metrics/gold_rm/adapter_config.json": "e323f8652b0d1b571163c21db0175ac32bee06bc48022b82cd1f7d7c1e94b8fd",
    "metrics/gold_rm/adapter_model.safetensors": "fbfa95e616bc88f6f17da81067390053c938e1e36448cf65b41499603ce2d704",
    "metrics/gold_rm/value_head.safetensors": "4feddd918c31985e644c143c21a49b6739731f6e2eb78e858801109196505f08",
}

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

for relative_path, expected_hash in EXPECTED_SHA256.items():
    asset_path = PROJECT_ROOT / relative_path
    assert asset_path.exists(), f"Missing asset: {relative_path}"
    actual_hash = sha256_file(asset_path)
    assert actual_hash == expected_hash, (relative_path, actual_hash, expected_hash)
print(f"Verified {len(EXPECTED_SHA256)} organizer assets by SHA-256")

In [ ]:
def read_jsonl(path: Path) -> list[dict]:
    with path.open(encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]

kid_adult_rows = read_jsonl(DATA_DIR / "kid_adult.jsonl")
good_bad_rows = read_jsonl(DATA_DIR / "good_bad.jsonl")
public_style_rows = read_jsonl(DATA_DIR / "public_test_style.jsonl")
public_quality_rows = read_jsonl(DATA_DIR / "public_test_quality.jsonl")

EXPECTED_SCHEMAS = {
    "kid_adult": (kid_adult_rows, 1489, {"prompt", "kid", "adult"}),
    "good_bad": (good_bad_rows, 2226, {"instruction", "chosen", "rejected"}),
    "public_style": (public_style_rows, 50, {"prompt", "kid", "adult", "fact"}),
    "public_quality": (public_quality_rows, 50, {"prompt", "chosen", "rejected"}),
}
for name, (rows, expected_count, expected_keys) in EXPECTED_SCHEMAS.items():
    assert len(rows) == expected_count, (name, len(rows), expected_count)
    assert all(set(row) == expected_keys for row in rows), f"Unexpected schema in {name}"
    assert all(isinstance(value, str) and value.strip() for row in rows for value in row.values())

assert not ({row["prompt"] for row in kid_adult_rows} & {row["prompt"] for row in public_style_rows})
quality_group_sizes = Counter(row["instruction"] for row in good_bad_rows)
print({name: len(rows) for name, (rows, _, _) in EXPECTED_SCHEMAS.items()})
print("Unique quality instructions:", len(quality_group_sizes))
print("Pairs per quality instruction:", Counter(quality_group_sizes.values()))
print("Public datasets are evaluation-only and are not included in any training variable.")

## 3. Формальные измерители

Style probability, pairwise reward accuracy и length-normalized implicit-preference accuracy.

## 4. Задача 1 — SFT

## 5. Задача 2 — DPO по стилю

## 6. Задача 3 — Reward Model

## 7. Задача 4 — DPO по качеству

## 8. Задача 5 — SimPO

## 9. Итоговые метрики

Финальная таблица должна печатать точные значения и буквы интервалов для всех пяти задач.